# 실패한 22종 추가 학습 — Kaggle/Roboflow 인증 확인 후 수정판\n\n기존 11개 학습 모델 유지, 원본은 `/content`에서만 사용, 학습된 모델만 Drive에 저장.\n\n첫 번째 셀을 먼저 실행하고, 각 운동 셀을 순서대로 실행하세요.

In [ ]:
# -*- coding: utf-8 -*-
"""Exercise AI: Retry only 22 missing exercise models in Google Colab.
Raw files and interim features: /content. Only trained model + tiny metadata: Google Drive.
GOOD labels inferred by the user for demonstrations and action-class datasets are NOT verified.
"""
import os, sys, re, json, time, hashlib, shutil, subprocess, importlib, zipfile, traceback
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

def ensure(module, package):
    try: importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
for mod, pkg in [('numpy','numpy'),('pandas','pandas'),('cv2','opencv-python-headless'),
                 ('PIL','Pillow'),('requests','requests'),('joblib','joblib'),
                 ('sklearn','scikit-learn'),('mediapipe','mediapipe>=0.10.30')]:
    ensure(mod,pkg)
import numpy as np, pandas as pd, cv2, requests, joblib, mediapipe as mp
from PIL import Image
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive')
except ImportError:
    userdata = None
    DRIVE = Path.cwd()

# Never touch the 11 existing trained models; this notebook has only the 22 missing tasks.
OUT = DRIVE/'Exercise_AI_Models'/'Equipment26'
TMP = Path('/content/ExerciseAI_22_RETRY_TEMP') if Path('/content').exists() else Path.cwd()/'ExerciseAI_22_RETRY_TEMP'
TMP.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('KAGGLEHUB_CACHE', str(TMP/'kagglehub_cache'))

def secret(key):
    x = os.environ.get(key,'')
    if x: return x
    if userdata is not None:
        try: return userdata.get(key) or ''
        except Exception: return ''
    return ''

# Strict matching: these are the 22 failed exercise keys, NOT other similar movements.
# Kaggle and Roboflow class aliases are specifically exercise names, not GOOD/BAD labels.
def M(ko, cat, aliases, kaggle=None, rf=None, exclude=()):
    return dict(ko=ko, category=cat, aliases=aliases,
                kaggle=kaggle or [], roboflow=rf or [], exclude=exclude)
TASKS = {
 'treadmill_walk':M('트레드밀 걷기','Cardio',['treadmill walking','treadmill walk','walking on treadmill']),
 'recumbent_cycling':M('리컴번트 바이크','Cardio',['recumbent bike','recumbent cycling','recumbent bicycle']),
 'upright_cycling':M('업라이트 바이크','Cardio',['upright cycling','stationary bike','bicycling stationary','stationary cycling']),
 'elliptical_training':M('일립티컬','Cardio',['elliptical trainer','elliptical machine','elliptical exercise']),
 'seated_chest_press':M('시티드 체스트 프레스','Chest',['seated chest press','lever chest press','machine chest press'],exclude=['incline','decline','fly']),
 'pec_deck_fly':M('펙덱 플라이','Chest',['pec deck fly','chest fly machine','lever pec deck fly','butterfly machine'], kaggle=['chest fly machine'],rf=['chest fly machine'],exclude=['reverse','dumbbell']),
 'seated_cable_row':M('롱 풀 시티드 로우','Back',['seated cable row','cable seated row','cable low row','seated low row'],exclude=['upright']),
 'machine_biceps_curl':M('머신 암 컬','Arms',['machine biceps curl','lever preacher curl','lever biceps curl','preacher curl machine'],exclude=['dumbbell','barbell']),
 'abdominal_crunch':M('앱도미널 크런치','Core',['machine abdominal crunch','lever seated crunch','ab crunch machine','abdominal crunch machine']),
 'rotary_torso':M('로터리 토르소','Core',['rotary torso','lever seated twist','torso rotation machine','seated torso twist']),
 'leg_extension':M('레그 익스텐션','Legs',['leg extension','lever leg extension','machine leg extension'], kaggle=['leg extension'],rf=['leg extension'],exclude=['curl']),
 'seated_leg_press':M('시티드 레그 프레스','Legs',['seated leg press','lever seated leg press','horizontal leg press'],exclude=['45 degree','sled']),
 'smith_squat':M('스미스 스쿼트','Legs',['smith machine squat','smith back squat','smith squat'],exclude=['dumbbell']),
 'double_twist':M('더블 트위스트','Core',['standing waist twist','waist twist machine','standing torso twist'],exclude=['russian twist']),
 'flat_bench_press':M('플랫 벤치프레스','Chest',['barbell bench press','flat bench press','barbell flat bench press','bench press'],kaggle=['bench press'],rf=['bench press'],exclude=['incline','decline','dumbbell','smith']),
 'incline_bench_press':M('인클라인 벤치프레스','Chest',['incline bench press','barbell incline bench press','incline barbell bench press'],kaggle=['incline bench press'],rf=['incline bench press'],exclude=['decline','dumbbell','fly','smith']),
 'decline_bench_press':M('디클라인 벤치프레스','Chest',['decline bench press','barbell decline bench press','decline barbell bench press'],kaggle=['decline bench press'],rf=['decline bench press'],exclude=['incline','dumbbell','fly','smith']),
 'machine_chest_press':M('머신 체스트 프레스','Chest',['machine chest press','seated chest press','lever chest press'],exclude=['incline','decline','fly']),
 'chest_fly':M('머신 체스트 플라이','Chest',['chest fly machine','pec deck fly','machine chest fly','butterfly machine'],kaggle=['chest fly machine'],rf=['chest fly machine'],exclude=['incline','cable','dumbbell']),
 'high_to_low_fly':M('하이 투 로우 케이블 플라이','Chest',['high to low cable fly','high cable fly','high cable crossover'],exclude=['low to high']),
 'low_to_high_fly':M('로우 투 하이 케이블 플라이','Chest',['low to high cable fly','low cable fly','low cable crossover'],exclude=['high to low']),
 'incline_dumbbell_fly':M('인클라인 덤벨 플라이','Chest',['incline dumbbell fly','incline dumbbell chest fly','dumbbell incline fly'],exclude=['press','decline','flat']),
}
assert len(TASKS)==22
TRAINED_11 = {'high_fly','machine_shoulder_press','chin_up','leg_curl','cable_crossover',
              'mega_max_lat_pulldown','dip_station_dip','power_leg_press','power_rack_squat',
              'sit_up','hyperextension'}
assert not (set(TASKS) & TRAINED_11)

MIN_UNIQUE_MEDIA = 8    # 8 independent source files, NOT GIF frames
MAX_GH_MEDIA_PER_TASK = 45
MAX_KG_FILES_PER_TASK = 35
MAX_RF_PER_CLASS = 180
MAX_TRAIN_MEDIA = 225
MAX_VIDEO_FRAMES = 18
MAX_MEDIA_BYTES = 240*1024*1024
MAX_RUN_GB = 3.5
results_22 = []

VIDEO_EXT = {'.mp4','.avi','.mov','.mkv','.webm','.m4v'}
IMAGE_EXT = {'.jpg','.jpeg','.png','.webp','.gif'}
ALL_EXT = VIDEO_EXT|IMAGE_EXT

def norm(s):
    return ' '.join(re.sub(r'[^a-z0-9]+',' ',str(s).lower()).split())

def exact_alias(text, aliases):
    t = norm(text)
    return t in {norm(x) for x in aliases}

def github_match(name, spec, equipment=''):
    title = norm(name)
    if not (any(title==norm(a) for a in spec['aliases'])): return False
    if any(norm(bad) in title for bad in spec['exclude']): return False
    return True

def media_size_mb(root):
    return sum(p.stat().st_size for p in root.rglob('*') if p.is_file())/(1024**2)

requests_session=requests.Session()
requests_session.headers.update({'User-Agent':'ExerciseAI-Educational-Research/2.0'})

def get_json(url):
    response=requests_session.get(url,timeout=50)
    response.raise_for_status()
    return response.json()

def retrieve(url,path,maxbytes=25*1024*1024):
    if not isinstance(url,str) or not url.startswith('https://'):return False
    if path.exists() and path.stat().st_size>1024:return True
    try:
        with requests_session.get(url,stream=True,timeout=50) as r:
            r.raise_for_status()
            n=0;path.parent.mkdir(parents=True,exist_ok=True)
            with path.open('wb') as out:
                for ch in r.iter_content(256*1024):
                    n+=len(ch)
                    if n>maxbytes:raise ValueError('file too large')
                    if ch:out.write(ch)
        return path.stat().st_size>1024
    except Exception:
        path.unlink(missing_ok=True)
        return False

_GH_INDEXES=None

def github_indexes():
    global _GH_INDEXES
    if _GH_INDEXES is not None:return _GH_INDEXES
    _GH_INDEXES=[]
    resources=[
      ('yuhonas','https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/dist/exercises.json',
       'https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/exercises/'),
      ('blackjyn','https://raw.githubusercontent.com/blackjyn/gym-exercises-dataset/main/data/exercises.json',
       'https://raw.githubusercontent.com/blackjyn/gym-exercises-dataset/main/'),
      ('namnq','https://cdn.jsdelivr.net/gh/namnq/workout-free-videos@v2.0.0/data/exercises.json',
       'https://cdn.jsdelivr.net/gh/namnq/workout-free-videos@v2.0.0/')]
    for name,uri,base in resources:
        try:
            blob=get_json(uri)
            if isinstance(blob,dict):blob=blob.get('exercises',blob.get('data',[]))
            if not isinstance(blob,list):raise ValueError('Not a list')
            _GH_INDEXES.append((name,base,blob))
            print(' GitHub',name,':',len(blob),'exercise records')
        except Exception as e:
            print(' GitHub',name,'index unavailable:',str(e)[:110])
    return _GH_INDEXES

def collect_github(key,where,entries):
    spec=TASKS[key];n=0;matched=0
    for ds,base,items in github_indexes():
        for it in items:
            if n>=MAX_GH_MEDIA_PER_TASK:break
            if not isinstance(it,dict) or not github_match(it.get('name',''),spec,it.get('equipment','')):continue
            matched+=1
            candidates=[]
            if ds=='yuhonas':
                candidates=[base+v.lstrip('/') for v in it.get('images',[]) if isinstance(v,str)]
            elif ds=='blackjyn':
                for field in ('gif_url','image','thumbnail'):
                    v=it.get(field,'')
                    if isinstance(v,dict):v=v.get('url','')
                    if isinstance(v,str) and v:
                        candidates.append(v if v.startswith('https://') else base+v.lstrip('/'))
            else:
                video=it.get('video',{})
                v=video.get('path','') if isinstance(video,dict) else ''
                if v: candidates.append(base+v.lstrip('/'))
            for url in candidates[:3]:
                ext=Path(url.split('?')[0]).suffix.lower()
                if ext not in ALL_EXT:continue
                path=where/(ds+'_'+hashlib.sha1(url.encode()).hexdigest()[:15]+ext)
                if retrieve(url,path,maxbytes=28*1024*1024):
                    entries.append((path,'ASSUMED_GOOD',f'GitHub:{ds}'))
                    n+=1
                if n>=MAX_GH_MEDIA_PER_TASK:break
    return {'site':'GitHub','status':'downloaded' if n else 'no_match','matched_records':matched,'files':n}

# Kaggle authenticates with the current API token OR a legacy username/key pair.
# Do NOT request or echo secrets. Only the known 22-class Workout/Fitness video dataset is used.
_KG_API=None;_KG_ENTRIES=None;_KG_ERROR=None
KG_HANDLE='hasyimabdillah/workoutfitness-video'

def kaggle_api_and_entries():
    global _KG_API,_KG_ENTRIES,_KG_ERROR
    if _KG_API is not None:return _KG_API,_KG_ENTRIES,None
    if _KG_ERROR:return None,None,_KG_ERROR
    tok=secret('KAGGLE_API_TOKEN');user=secret('KAGGLE_USERNAME');key=secret('KAGGLE_KEY')
    if tok:os.environ['KAGGLE_API_TOKEN']=tok
    if user and key:
        os.environ['KAGGLE_USERNAME']=user
        os.environ['KAGGLE_KEY']=key
    if not (tok or (user and key)):
        _KG_ERROR='Kaggle credentials not registered';return None,None,_KG_ERROR
    try:
        ensure('kaggle','kaggle>=2.0.0')
        from kaggle.api.kaggle_api_extended import KaggleApi
        api=KaggleApi();api.authenticate()
        # Works on both older and newer Kaggle clients.
        allfiles=[];seen=set();nexttoken=None
        for _ in range(70):
            if nexttoken:
                try: response=api.dataset_list_files(KG_HANDLE,page_token=nexttoken)
                except TypeError:break
            else:response=api.dataset_list_files(KG_HANDLE)
            for f in getattr(response,'files',[]) or []:
                name=str(getattr(f,'name',''))
                if name and name not in seen:
                    allfiles.append((name,int(getattr(f,'total_bytes',0) or 0)))
                    seen.add(name)
            fresh=getattr(response,'next_page_token',None) or getattr(response,'nextPageToken',None)
            if not fresh or fresh==nexttoken:break
            nexttoken=fresh
        _KG_API,_KG_ENTRIES=api,allfiles
        print(' Kaggle authenticated; listing contains',len(allfiles),'files')
        return api,allfiles,None
    except BaseException as e:
        _KG_ERROR=f'{type(e).__name__}: {str(e)[:320]}'
        print(' Kaggle connection error:',_KG_ERROR)
        return None,None,_KG_ERROR

def kaggle_path_match(remote,aliases):
    p=Path(remote)
    if any(exact_alias(part,aliases) for part in p.parts[:-1]):return True
    name=norm(p.stem)
    return any(name==norm(a) or name.startswith(norm(a)+' ') for a in aliases)

def collect_kaggle(key,where,entries):
    spec=TASKS[key]
    # More than six sports can use dataset search; still reject ambiguous exercise labels.
    search_aliases=spec['kaggle'] or spec['aliases'][:2]
    api,files,error=kaggle_api_and_entries()
    if error:return {'site':'Kaggle','status':'AUTH_OR_LIST_ERROR','reason':error}
    direct=[(name,size) for name,size in files if Path(name).suffix.lower() in VIDEO_EXT|IMAGE_EXT and kaggle_path_match(name,search_aliases)]
    count=0;errs=[]
    for remote,size in direct[:MAX_KG_FILES_PER_TASK]:
        if size>MAX_MEDIA_BYTES or media_size_mb(TMP)>MAX_RUN_GB*1024:continue
        folder=where/('file_'+hashlib.sha1(remote.encode()).hexdigest()[:12]);folder.mkdir(parents=True,exist_ok=True)
        try:
            api.dataset_download_file(KG_HANDLE,remote,path=str(folder),quiet=True,force=False)
            items=list(folder.rglob('*'))
            for z in [x for x in items if x.suffix.lower()=='.zip']:
                with zipfile.ZipFile(z) as archive:
                    for inf in archive.infolist():
                        if Path(inf.filename).suffix.lower() not in VIDEO_EXT|IMAGE_EXT or inf.file_size>MAX_MEDIA_BYTES:continue
                        dest=folder/(hashlib.sha1(inf.filename.encode()).hexdigest()[:12]+Path(inf.filename).suffix.lower())
                        with archive.open(inf) as inp,dest.open('wb') as out:shutil.copyfileobj(inp,out)
            for f in folder.rglob('*'):
                if f.suffix.lower() in VIDEO_EXT|IMAGE_EXT and f.stat().st_size>1024:
                    entries.append((f,'ASSUMED_GOOD','Kaggle:WorkoutFitnessVideo'));count+=1
        except BaseException as e:errs.append(f'{remote}:{str(e)[:110]}')
    # If the known dataset has no matching files, actively discover other public datasets
    # without guessing that they contain labeled GOOD or BAD.
    discovered=[]
    if count==0:
        for phrase in search_aliases[:2]:
            try:
                for ds in list(api.dataset_list(search=phrase,page=1))[:8]:
                    ref=str(getattr(ds,'ref',''))
                    if not ref or ref==KG_HANDLE or ref in discovered:continue
                    discovered.append(ref)
                    try:
                        dsfiles=api.dataset_list_files(ref)
                        candidates=[str(x.name) for x in (getattr(dsfiles,'files',[]) or [])
                                    if Path(str(x.name)).suffix.lower() in VIDEO_EXT|IMAGE_EXT and
                                    kaggle_path_match(str(x.name),search_aliases)]
                        for remote in candidates[:8]:
                            if count>=MAX_KG_FILES_PER_TASK or media_size_mb(TMP)>MAX_RUN_GB*1024:break
                            folder=where/('extra_'+hashlib.sha1((ref+remote).encode()).hexdigest()[:12]);folder.mkdir(parents=True,exist_ok=True)
                            try:
                                api.dataset_download_file(ref,remote,path=str(folder),quiet=True,force=False)
                                for z in folder.rglob('*.zip'):
                                    with zipfile.ZipFile(z) as arc:
                                        for inf in arc.infolist():
                                            if Path(inf.filename).suffix.lower() not in VIDEO_EXT|IMAGE_EXT or inf.file_size>MAX_MEDIA_BYTES:continue
                                            dest=folder/(hashlib.sha1(inf.filename.encode()).hexdigest()[:12]+Path(inf.filename).suffix.lower())
                                            with arc.open(inf) as inp,dest.open('wb') as out:shutil.copyfileobj(inp,out)
                                for f in folder.rglob('*'):
                                    if f.suffix.lower() in VIDEO_EXT|IMAGE_EXT and f.stat().st_size>1024:
                                        entries.append((f,'ASSUMED_GOOD','Kaggle:'+ref));count+=1
                            except BaseException as e:errs.append(f'{ref}/{remote}:{str(e)[:100]}')
                    except BaseException as e:errs.append(f'listing {ref}:{str(e)[:100]}')
            except BaseException as e:errs.append(f'search {phrase}:{str(e)[:100]}')
    return {'site':'Kaggle','status':'DOWNLOADED' if count else 'NO_MATCHING_MEDIA',
            'known_dataset_matches':len(direct),'discovered_datasets':discovered[:15],
            'files':count,'errors':errs[:4]}

# Roboflow classification dataset: these are EXERCISE CLASS names, not quality labels.
# Download once to /content; reuse across all 22 model cells; no raw data goes to Drive.
_RF_ROOT=None;_RF_ERROR=None
RF_WS='first-easq8';RF_PROJ='exercise-detection-crmtf';RF_VERSION=1

def roboflow_folder():
    global _RF_ROOT,_RF_ERROR
    if _RF_ROOT is not None:return _RF_ROOT,None
    if _RF_ERROR:return None,_RF_ERROR
    key=secret('ROBOFLOW_API_KEY')
    if not key:return None,'ROBOFLOW_API_KEY missing'
    folder=TMP/'roboflow_exercise_classification'
    try:
        ensure('roboflow','roboflow')
        from roboflow import Roboflow
        rf=Roboflow(api_key=key)
        project=rf.workspace(RF_WS).project(RF_PROJ)
        errors=[]
        for fmt in ('folder','multiclass','classification'):
            try:
                ds=project.version(RF_VERSION).download(fmt,location=str(folder),overwrite=False)
                root=Path(getattr(ds,'location',folder))
                if any(root.rglob('*.jpg')) or any(root.rglob('*.png')):
                    _RF_ROOT=root
                    print(' Roboflow classification downloaded:',root,'format',fmt)
                    return root,None
                errors.append(fmt+': no image files extracted')
            except Exception as e:errors.append(fmt+': '+str(e)[:160])
        _RF_ERROR=' | '.join(errors)[:450]
        print(' Roboflow download unsuccessful:',_RF_ERROR)
        return None,_RF_ERROR
    except Exception as e:
        _RF_ERROR=f'{type(e).__name__}: {str(e)[:320]}'
        print(' Roboflow connection error:',_RF_ERROR)
        return None,_RF_ERROR

def collect_roboflow(key,where,entries):
    spec=TASKS[key]
    # The verified public classification dataset has ONLY the listed exercise classes.
    # Other exercises must not be silently merged with approximate labels.
    if not spec['roboflow']:return {'site':'Roboflow','status':'NO_CLASS_FOR_THIS_EXERCISE'}
    root,err=roboflow_folder()
    if err:return {'site':'Roboflow','status':'DOWNLOAD_ERROR','reason':err}
    hits=[];classes=[]
    for parent in [root/'train',root/'valid',root/'test',root]:
        if not parent.is_dir():continue
        for d in parent.iterdir():
            if not d.is_dir():continue
            classes.append(d.name)
            if exact_alias(d.name,spec['roboflow']):
                hits += [f for f in d.rglob('*') if f.suffix.lower() in IMAGE_EXT]
    hits=list(dict.fromkeys(hits))
    if not hits:
        return {'site':'Roboflow','status':'CLASS_NOT_FOUND','expected':spec['roboflow'],
                'available_sample':sorted(set(classes))[:35]}
    if len(hits)>MAX_RF_PER_CLASS:
        hits=[hits[int(i)] for i in np.linspace(0,len(hits)-1,MAX_RF_PER_CLASS,dtype=int)]
    for f in hits:entries.append((f,'ASSUMED_GOOD','Roboflow:exercise-detection-crmtf/1'))
    return {'site':'Roboflow','status':'AVAILABLE','files':len(hits)}

# Mendeley and Papers With Code provide research/index information, but no verified
# matching 22-exercise GOOD/BAD RGB media download endpoint was established.
# Do not pretend that these sources were trained.
REFERENCES={
 'Mendeley':['https://data.mendeley.com/datasets/kgbb3yn47p/3',
             'https://data.mendeley.com/datasets/j53wpz5sf5/1'],
 'Papers With Code':['https://paperswithcode.com/dataset/ucf50',
                     'https://github.com/nxc1802/Gym_Classification']
}

def frame_iterator(path):
    ext=path.suffix.lower()
    if ext in IMAGE_EXT and ext!='.gif':
        img=cv2.imread(str(path))
        if img is not None:yield img
    elif ext=='.gif':
        try:
            with Image.open(path) as im:
                n=im.n_frames if hasattr(im,'n_frames') else 1
                for at in sorted(set(np.linspace(0,n-1,min(5,n),dtype=int))):
                    im.seek(int(at))
                    img=cv2.cvtColor(np.asarray(im.convert('RGB')),cv2.COLOR_RGB2BGR)
                    yield img
        except Exception:pass
    elif ext in VIDEO_EXT:
        cap=cv2.VideoCapture(str(path))
        try:
            n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if n<3:return
            for at in sorted(set(np.linspace(0,n-1,MAX_VIDEO_FRAMES,dtype=int))):
                cap.set(cv2.CAP_PROP_POS_FRAMES,int(at));ok,img=cap.read()
                if ok:yield img
        finally:cap.release()

POSE_ASSET=TMP/'pose_landmarker_lite.task'
POSE_URL='https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'

def pose_asset():
    if not POSE_ASSET.exists() or POSE_ASSET.stat().st_size<1000000:
        r=requests_session.get(POSE_URL,timeout=120);r.raise_for_status()
        POSE_ASSET.write_bytes(r.content)
    return POSE_ASSET

LANDMARK_IDS=[11,12,13,14,15,16,23,24,25,26,27,28,0]
FEATURE_DIM=len(LANDMARK_IDS)*2*3+1  # mean / std / 10-90 percentile range + valid fraction

def frame_pose_features(frame,detector):
    if frame is None or frame.shape[0]<35 or frame.shape[1]<35:return None
    h,w=frame.shape[:2];scale=min(1.,720/max(h,w))
    if scale<1:frame=cv2.resize(frame,(int(w*scale),int(h*scale)))
    rgb=np.ascontiguousarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
    result=detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))
    if not result.pose_landmarks:return None
    lm=result.pose_landmarks[0]
    visibility=np.array([float(lm[i].visibility or 0) for i in LANDMARK_IDS],dtype=np.float32)
    if np.count_nonzero(visibility[:12]>0.25)<5:return None
    xy=np.array([[lm[i].x,lm[i].y] for i in LANDMARK_IDS],dtype=np.float32)
    if not np.isfinite(xy).all():return None
    hip=(xy[6]+xy[7])/2
    shoulder=(xy[0]+xy[1])/2
    size=max(float(np.linalg.norm(hip-shoulder)),float(np.linalg.norm(xy[0]-xy[1])),0.06)
    coords=(xy-shoulder)/size
    return np.clip(coords.flatten(),-10,10)

def file_pose_features(path,detector):
    vals=[];frames=0
    for frame in frame_iterator(path):
        frames+=1
        try:
            feature=frame_pose_features(frame,detector)
            if feature is not None:vals.append(feature)
        except Exception:continue
    if not vals:return None,0
    data=np.asarray(vals,dtype=np.float32)
    mean=data.mean(axis=0)
    sd=data.std(axis=0)
    spread=np.percentile(data,90,axis=0)-np.percentile(data,10,axis=0)
    ratio=np.array([len(vals)/max(1,frames)],dtype=np.float32)
    return np.nan_to_num(np.concatenate([mean,sd,spread,ratio]),nan=0,posinf=0,neginf=0),len(vals)

def dhash_file(path):
    # A perceptual image hash avoids counting exact/near-identical downloaded photos
    # from mirrored data sources as new independent examples.
    try:
        if path.suffix.lower() in VIDEO_EXT:
            return None
        if path.suffix.lower()=='.gif':
            with Image.open(path) as im:gray=np.asarray(im.convert('L').resize((9,8)))
        else:
            img=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
            if img is None:return None
            gray=cv2.resize(img,(9,8))
        bits=(gray[:,1:]>gray[:,:-1]).flatten()
        return int(''.join('1' if b else '0' for b in bits),2)
    except Exception:return None

def unique_media(entries):
    # SHA256 dedupe; dHash suppresses very similar static screenshots.
    output=[];seen=set();seen_dh=[]
    for p,label,source in entries:
        if not p.is_file() or p.suffix.lower() not in ALL_EXT:continue
        try:
            if p.stat().st_size<1000:continue
            # Streaming hash avoids loading big videos into RAM.
            digest=hashlib.sha256()
            with p.open('rb') as f:
                for ch in iter(lambda:f.read(1024*1024),b''):digest.update(ch)
            hexd=digest.hexdigest()
            if hexd in seen:continue
            seen.add(hexd)
            dh=dhash_file(p)
            if dh is not None and any((dh^old).bit_count()<=3 for old in seen_dh):continue
            if dh is not None:seen_dh.append(dh)
            output.append((p,label,source))
        except Exception:continue
    return output


def train_one(key):
    if key not in TASKS:raise KeyError(key)
    spec=TASKS[key];name=spec['ko'];start=time.monotonic()
    modelpath=OUT/spec['category']/(key+'.joblib')
    modelinfo=modelpath.with_suffix('.json')
    result={'exercise':key,'ko':name,'status':'UNKNOWN','model_path':None,
            'sources':[],'unique_media_files':0,'pose_samples':0,
            'possible_license_restrictions':['check source license before reuse'],
            'index_only':REFERENCES}
    print('\n'+'='*72+'\n',key,'—',name)
    if modelpath.exists():
        print('[기존 모델 유지] 재학습하지 않음:',modelpath)
        result.update(status='SKIPPED_EXISTING_MODEL',model_path=str(modelpath))
        results_22.append(result)
        return result
    folder=TMP/'train_data'/key;folder.mkdir(parents=True,exist_ok=True)
    data=[]
    for collector,args in [(collect_github,(key,folder/'github',data)),
                           (collect_kaggle,(key,folder/'kaggle',data)),
                           (collect_roboflow,(key,folder/'roboflow',data))]:
        try:
            (args[1] if len(args)>1 else folder).mkdir(parents=True,exist_ok=True)
            source_result=collector(*args)
            result['sources'].append(source_result)
            print(' ',source_result['site'],source_result['status'],source_result.get('files',''),str(source_result.get('reason',source_result.get('errors','')))[:200])
        except Exception as ex:
            result['sources'].append({'site':collector.__name__,'status':'ERROR','reason':str(ex)[:260]})
            print(' ',collector.__name__,'error:',str(ex)[:180])
    # Optional manual reviewed media (in Colab temporary, not in Drive).
    for label in ('GOOD','BAD'):
        manual=Path('/content/exercise_ai_22_reviewed')/key/label
        if manual.exists():
            for p in manual.rglob('*'):
                if p.suffix.lower() in ALL_EXT:
                    data.append((p,label,'MANUALLY_REVIEWED'))
    data=unique_media(data)
    result['unique_media_files']=len(data)
    if not data:
        result['status']='NOT_TRAINED_NO_DATA'
        print(' DATA NOT AVAILABLE — no model created')
        results_22.append(result)
        return result
    # Prefer actual MP4 video items where there is enough independent material.
    videos=[r for r in data if r[0].suffix.lower() in VIDEO_EXT]
    images=[r for r in data if r[0].suffix.lower() in IMAGE_EXT]
    selected=videos if len(videos)>=MIN_UNIQUE_MEDIA else images if len(images)>=MIN_UNIQUE_MEDIA else data
    selected=selected[:MAX_TRAIN_MEDIA]
    result['media_mode']='video' if selected==videos else 'image' if selected==images else 'mixed'
    try:pose_asset()
    except Exception as ex:
        result['status']='NOT_TRAINED_POSE_MODEL_UNAVAILABLE'
        result['error']=str(ex)[:240];results_22.append(result);return result
    opts=mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(POSE_ASSET)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,num_poses=1,
        min_pose_detection_confidence=0.40,min_pose_presence_confidence=0.40)
    sample=[];lab=[];origin=[];counts=Counter()
    with mp.tasks.vision.PoseLandmarker.create_from_options(opts) as detector:
        for i,(file,label,source) in enumerate(selected,1):
            try:
                f,n=file_pose_features(file,detector)
                if f is not None and len(f)==FEATURE_DIM:
                    sample.append(f);lab.append(label);origin.append(source);counts[source]+=1
            except Exception:continue
    X=np.asarray(sample,dtype=np.float32)
    result['pose_samples']=len(sample)
    result['source_samples']=dict(counts)
    result['labeled_media']=dict(Counter(lab))
    if len(sample)<MIN_UNIQUE_MEDIA:
        result['status']='NOT_TRAINED_INSUFFICIENT_POSES'
        print(' INSUFFICIENT:',len(sample),'valid source files (minimum',MIN_UNIQUE_MEDIA,')')
        results_22.append(result)
        return result
    yy=np.asarray(lab)
    # GOOD is always a user assumption for public demonstrations/action datasets;
    # only manually reviewed files can carry a verified BAD label.
    if np.sum(yy=='BAD')>=8 and np.sum(np.isin(yy,['ASSUMED_GOOD','GOOD']))>=8:
        binary=np.where(yy=='BAD','BAD','GOOD')
        model=make_pipeline(StandardScaler(),RandomForestClassifier(
            n_estimators=300,class_weight='balanced_subsample',random_state=42,n_jobs=-1,min_samples_leaf=2))
        model.fit(X,binary)
        kind='TRAINED_GOOD_BAD'
        result['quality_note']='BAD samples explicitly reviewed; public GOOD assumed, not verified'
        threshold=None
    else:
        valid=(yy!='BAD')
        if int(np.sum(valid))<MIN_UNIQUE_MEDIA:
            result['status']='NOT_TRAINED_INSUFFICIENT_GOOD'
            results_22.append(result);return result
        model=make_pipeline(StandardScaler(),IsolationForest(n_estimators=300,
                               contamination=0.1,random_state=42,n_jobs=-1))
        model.fit(X[valid])
        kind='TRAINED_GOOD_ONLY'
        # For interpretation only. Not a calibrated probability of bad form.
        scores=model.score_samples(X[valid]);threshold=float(np.quantile(scores,0.10))
        result['quality_note']='GOOD assumed from action class/demo, BAD means anomalous vs that distribution, not verified exercise error'
    modelpath.parent.mkdir(parents=True,exist_ok=True)
    payload={'model':model,'exercise':key,'exercise_name':name,'mode':kind,
             'feature_spec':'13 pose joints shoulder-hip normalized xy; aggregate mean,std,p90-p10,valid_fraction v1',
             'feature_dim':FEATURE_DIM,'threshold':threshold,'media_mode':result['media_mode'],
             'source_file_counts':dict(counts),
             'assumptions':'Public action-class/demo sources use unverified assumed GOOD labels; no real BAD posture validation'}
    joblib.dump(payload,modelpath)
    result.update(status=kind,model_path=str(modelpath))
    result['elapsed_seconds']=round(time.monotonic()-start,2)
    meta={k:result.get(k) for k in ('exercise','ko','status','media_mode','unique_media_files','pose_samples','labeled_media','source_samples','quality_note','model_path','elapsed_seconds')}
    # User requested ONLY model result + minimal reproducibility metadata on Drive.
    modelinfo.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
    print(' SAVED:',modelpath,'| pose/source-files:',len(sample),'|',kind)
    results_22.append(result)
    return result


def train_safe(key):
    try:return train_one(key)
    except Exception as exc:
        item={'exercise':key,'status':'ERROR','message':f'{type(exc).__name__}: {exc}',
              'unique_media_files':0,'pose_samples':0,'model_path':None}
        results_22.append(item)
        print(' TRAINING ERROR:',key,repr(exc))
        traceback.print_exc(limit=2)
        return item

def show_summary():
    rows=[]
    for x in results_22:
        rows.append({k:x.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})
    table=pd.DataFrame(rows)
    if len(table):display(table)
    print('RETRY TASKS',len(TASKS),'| executed',len(rows), '| trained',
          sum(x.get('status','').startswith('TRAINED_') for x in results_22),
          '| not trained',sum(not x.get('status','').startswith('TRAINED_') for x in results_22))
    return table

def cleanup_temporary():
    # Keep model artifacts on Drive. Delete only fresh temporary data in /content.
    if TMP.exists() and TMP.resolve()!=OUT.resolve():
        shutil.rmtree(TMP)
        print('Removed /content temporary downloads; models on Google Drive preserved.')

print('Setup complete:',len(TASKS),'retry targets; existing 11 models untouched.')
print('Saved models only ->', OUT)
print('Kaggle auth: existing KAGGLE_USERNAME + KAGGLE_KEY work; token optional')
print('Roboflow auth (optional): Colab secret ROBOFLOW_API_KEY')


## 01/22 — treadmill_walk

In [ ]:
r01 = train_safe('treadmill_walk')
print({k:r01.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 02/22 — recumbent_cycling

In [ ]:
r02 = train_safe('recumbent_cycling')
print({k:r02.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 03/22 — upright_cycling

In [ ]:
r03 = train_safe('upright_cycling')
print({k:r03.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 04/22 — elliptical_training

In [ ]:
r04 = train_safe('elliptical_training')
print({k:r04.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 05/22 — seated_chest_press

In [ ]:
r05 = train_safe('seated_chest_press')
print({k:r05.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 06/22 — pec_deck_fly

In [ ]:
r06 = train_safe('pec_deck_fly')
print({k:r06.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 07/22 — seated_cable_row

In [ ]:
r07 = train_safe('seated_cable_row')
print({k:r07.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 08/22 — machine_biceps_curl

In [ ]:
r08 = train_safe('machine_biceps_curl')
print({k:r08.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 09/22 — abdominal_crunch

In [ ]:
r09 = train_safe('abdominal_crunch')
print({k:r09.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 10/22 — rotary_torso

In [ ]:
r10 = train_safe('rotary_torso')
print({k:r10.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 11/22 — leg_extension

In [ ]:
r11 = train_safe('leg_extension')
print({k:r11.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 12/22 — seated_leg_press

In [ ]:
r12 = train_safe('seated_leg_press')
print({k:r12.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 13/22 — smith_squat

In [ ]:
r13 = train_safe('smith_squat')
print({k:r13.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 14/22 — double_twist

In [ ]:
r14 = train_safe('double_twist')
print({k:r14.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 15/22 — flat_bench_press

In [ ]:
r15 = train_safe('flat_bench_press')
print({k:r15.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 16/22 — incline_bench_press

In [ ]:
r16 = train_safe('incline_bench_press')
print({k:r16.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 17/22 — decline_bench_press

In [ ]:
r17 = train_safe('decline_bench_press')
print({k:r17.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 18/22 — machine_chest_press

In [ ]:
r18 = train_safe('machine_chest_press')
print({k:r18.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 19/22 — chest_fly

In [ ]:
r19 = train_safe('chest_fly')
print({k:r19.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 20/22 — high_to_low_fly

In [ ]:
r20 = train_safe('high_to_low_fly')
print({k:r20.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 21/22 — low_to_high_fly

In [ ]:
r21 = train_safe('low_to_high_fly')
print({k:r21.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 22/22 — incline_dumbbell_fly

In [ ]:
r22 = train_safe('incline_dumbbell_fly')
print({k:r22.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})


## 전체 결과 확인

In [ ]:
final_table = show_summary()


## 선택: 원본 임시 파일 삭제 (모델은 유지)

In [ ]:
cleanup_temporary()
